# Chapter 9: The Transformer Block

[Read this chapter online](https://jackluu.io/book/section-3-the-transformer/ch09-transformer-block/) &nbsp;|&nbsp; [Open in Colab](https://colab.research.google.com/github/jackluucoding/build-llm-from-zero/blob/main/notebooks/ch09-transformer-block.ipynb)

From *Building an LLM from Zero: Look Inside the Black Box* by Truong (Jack) Luu.


In [ ]:
# Run me first. Safe to run more than once; it skips whatever is already done.
import os
import subprocess
import sys

FOLDER = "build-llm-from-zero"

# 1. Fetch the code, unless we are already inside it
if os.path.basename(os.getcwd()) != FOLDER:
    if not os.path.isdir(FOLDER):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackluucoding/build-llm-from-zero"], check=True)
    os.chdir(FOLDER)
sys.path.insert(0, os.getcwd())

# 2. PyTorch, the CPU build, which is all this book needs
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)

# 3. The Shakespeare text
subprocess.run([sys.executable, "src/utils/download_data.py"], check=True)

print("Ready. Working in", os.getcwd())

# Chapter 9: The Transformer Block

![Where we are in the pipeline](../assets/diagrams/ch09-where-we-are.png){ width="756" }
*Figure 9.1: We combine our pieces into a reusable building block.*

After processing information with feed-forward and normalizing the math in Chapter 8, we now have all the ingredients: multi-head attention (where tokens talk to each other), feed-forward layers (where tokens think for themselves), and LayerNorm (to keep training stable). In this chapter you will:

* Combine these parts into a single Lego brick called a Transformer block.
* Add residual connections so deep networks can learn effectively.
* Stack multiple blocks to build depth.

**Words to Know**
    - **Transformer Block**: A reusable block of code containing attention, feed-forward, and normalization.
    - **Residual Connection**: A shortcut that lets information skip a step, keeping the original signal intact.

## Theory

### Putting the Pieces Together

A single Transformer block combines our tools into a powerful unit. The forward pass of one block is just two simple lines of code:

```python
x = x + self.attn(self.ln1(x))  # "listen, then add to what I know"
x = x + self.ff(self.ln2(x))    # "think, then update what I know"
```

Notice the `x = x + ...` pattern. This is the secret to making deep neural networks work.

### Trick 1: The Residual Connection

Instead of completely replacing a token's data with the output of the attention layer, we **add** the new information to the original data. This is called a **residual connection** (or skip connection), as seen in Figure 9.2.

![The data flows down a main highway, taking side trips for attention and feed-forward](../assets/diagrams/ch09-transformer-block-connections.png){ width="298" }
*Figure 9.2: Residual connections provide a direct highway through the block.*

Why does this matter? Imagine you are in a game of telephone. Each person translates the message and passes it on. After 10 rounds, the original message is usually unrecognizable. Residual connections are like passing the original written message alongside the game of telephone. Even if the spoken message gets mangled, the original is still there.

During training, feedback signals travel backward through the layers to adjust weights. Without residual connections, this feedback must pass through every layer in sequence. If each layer distorts the signal slightly, the signal either grows too large or shrinks to nearly nothing. The direct `+x` highway allows feedback to travel cleanly through dozens of stacked layers.

### Trick 2: Pre-Layer Norm

Notice that we apply LayerNorm *before* each major step: `attention(LayerNorm(x))`. 

This is the modern "pre-norm" convention used in GPT models (Figure 9.3). We normalize the data first, then do the hard work. It ensures that attention and feed-forward always receive well-behaved numbers, stabilizing training in the early stages.

![Data passes through LayerNorm before entering the Attention layer](../assets/diagrams/ch09-pre-norm.png){ width="498" }
*Figure 9.3: Normalize the data before the hard work.*

### Stacking Blocks

One Transformer block is powerful, but not enough. We stack multiple blocks in sequence (Figure 9.4). Our model uses 4 blocks.

Each block refines the token representations further. Think of it like reading a complex document multiple times. First pass: "who are the characters?" Second pass: "what are their motivations?" Third pass: "what are the themes?" Earlier blocks capture simple patterns, while later blocks capture deeper meaning. This is why deeper models perform better.

![Multiple blocks stacked on top of each other, building deeper understanding](../assets/diagrams/ch09-stacking-blocks.png){ width="338" }
*Figure 9.4: Stacking blocks allows the model to find complex patterns.*

Returning to the big picture map in Figure 9.1, these stacked Transformer blocks form the core engine of our model, ready to process the embeddings into deep, contextualized representations.

## Code

```python
class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.attn = MultiHeadAttention()
        self.ff   = FeedForward()
        # Layer normalization applied before attention and feed-forward
        self.ln1  = nn.LayerNorm(config.n_embd)
        self.ln2  = nn.LayerNorm(config.n_embd)

    def forward(self, x):
        # The + creates a residual connection, adding new info
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x
```

Run the script to see a block process data and to check the parameter count.

```python
$ python src/ch08_transformer_block.py
One TransformerBlock parameters: 197,888
  MultiHeadAttention : 65,664
  FeedForward        : 131,712
  LayerNorms (x2)    : 512

Input  shape: torch.Size([2, 10, 128])
Output shape: torch.Size([2, 10, 128])   (same as input)

--- Residual connection demonstration ---
The input is never lost -- it always flows through.

Original token norm  : 11.470
Attention output norm: 4.599
After residual norm  : 12.903  (combined)

--- Stacking multiple blocks ---
Stacking 4 blocks:
  Params per block: 197,888
  Total params    : 791,552  (4 x 197,888)

Input  shape: torch.Size([2, 10, 128])
Output shape: torch.Size([2, 10, 128])  (unchanged after 4 blocks)
```

**What just happened:**

1. Lines 4 through 8 build a block containing attention, feed-forward, and LayerNorm.
2. Line 12 shows the residual connection combine the original token data with the attention output.
3. We stacked 4 blocks and saw the data flow through smoothly.

**Shape Check:**

- Input to TransformerBlock: `[Batch, Time, 128]`
- Output of TransformerBlock: `[Batch, Time, 128]`

The block is **shape-preserving**. The input and output have identical shapes, which is exactly what allows us to stack them endlessly like Lego bricks.

## Try It

**Try It**
    Open `src/ch08_transformer_block.py` and change the number of layers in the stack from `config.n_layers` to `12`. Run the script again. Notice how the total parameter count grows, but the output shape remains exactly the same. 

**In Business**
    When building software systems (like a house-style writing assistant), you want modular, scalable processes. A Transformer block is the ultimate modular unit. If your assistant is not smart enough, you do not have to invent a new architecture; you just stack more blocks and train it longer.

## Key Takeaways

* A Transformer block combines LayerNorm, attention, and feed-forward layers.
* Residual connections (`x + ...`) create a direct highway for data, allowing deep models to learn effectively.
* Pre-norm applies LayerNorm before the hard work, stabilizing the math.
* Blocks preserve the shape of the data, allowing us to stack them modularly.

## Check Your Understanding

1. Why do we add the attention output to the original input (`x + attention`)?
2. What does "pre-norm" mean?
3. Why does the Transformer block output exactly the same shape it took in?


## Further Reading

**The shortcut that makes depth possible.** Stacking more layers used to make networks worse, not better, because the learning signal degraded on the way down. The fix was to add the input of a block back onto its output, giving the signal a clear path through. It was shown on image models, and every Transformer block, including the one in Chapter 9, uses it.

**The architecture this book builds.** Reading a sequence one step at a time is slow, because step 500 cannot start until step 499 has finished, and distant words stay hard to connect. This paper removed the step-by-step reading entirely and kept only attention, plus a note of each token's position. Every token can then be processed at once, which is what made training on very large amounts of text practical. The model you build in Chapters 6 to 10 is this design, made small.

<div class="refs" markdown>

He, K., Zhang, X., Ren, S., & Sun, J. (2015). *Deep residual learning for image recognition* (arXiv:1512.03385). arXiv. https://doi.org/10.48550/arXiv.1512.03385

Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, L., & Polosukhin, I. (2017). *Attention is all you need* (arXiv:1706.03762). arXiv. https://doi.org/10.48550/arXiv.1706.03762

</div>

---

### `src/ch08_transformer_block.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/ch08_transformer_block.py"   # a cell has none, and the file uses it to find the text

"""
Combine attention and feed-forward into a transformer block.
This file belongs to Chapter 9.
Run: python src/ch08_transformer_block.py
"""
import torch
import torch.nn as nn
import os
import sys

from src.utils.config import GPTConfig
from src.ch06_multihead_attention import MultiHeadAttention
from src.ch07_feedforward import FeedForward

# Settings
config = GPTConfig()

# --- The Idea ---
class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.attn = MultiHeadAttention()
        self.ff   = FeedForward()
        # Layer normalization applied before attention and feed-forward
        self.ln1  = nn.LayerNorm(config.n_embd)
        self.ln2  = nn.LayerNorm(config.n_embd)

    def forward(self, x):
        # The + creates a residual connection, adding new info
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

# --- Demo ---
if __name__ == "__main__":
    torch.manual_seed(42)
    print("Chapter 9: The Transformer Block\n")

    block = TransformerBlock()
    total_params = sum(p.numel() for p in block.parameters())
    attn_params  = sum(p.numel() for p in block.attn.parameters())
    ff_params    = sum(p.numel() for p in block.ff.parameters())
    ln_params    = sum(p.numel() for p in block.ln1.parameters()) + \
                   sum(p.numel() for p in block.ln2.parameters())

    print(f"One TransformerBlock parameters: {total_params:,}")
    print(f"  MultiHeadAttention : {attn_params:,}")
    print(f"  FeedForward        : {ff_params:,}")
    print(f"  LayerNorms (x2)    : {ln_params:,}")

    B, T = 2, 10
    x = torch.randn(B, T, config.n_embd)
    out = block(x)
    print(f"\nInput  shape: {x.shape}")
    print(f"Output shape: {out.shape}   (same as input)")

    print("\n--- Residual connection demonstration ---")
    print("The input is never lost -- it always flows through.")

    with torch.no_grad():
        x_sample  = x[0, 0, :]
        attn_out  = block.attn(block.ln1(x[0:1]))[0, 0, :]
        final_out = block(x[0:1])[0, 0, :]

    print(f"\nOriginal token norm  : {x_sample.norm():.3f}")
    print(f"Attention output norm: {attn_out.norm():.3f}")
    print(f"After residual norm  : {final_out.norm():.3f}  (combined)")

    print("\n--- Stacking multiple blocks ---")
    n_layers = config.n_layers
    blocks = nn.Sequential(*[TransformerBlock() for _ in range(n_layers)])
    total_stack_params = sum(p.numel() for p in blocks.parameters())

    print(f"Stacking {n_layers} blocks:")
    print(f"  Params per block: {total_params:,}")
    print(
        f"  Total params    : {total_stack_params:,}  "
        f"({n_layers} x {total_params:,})"
    )

    x = torch.randn(B, T, config.n_embd)
    out = blocks(x)
    print(f"\nInput  shape: {x.shape}")
    print(f"Output shape: {out.shape}  (unchanged after {n_layers} blocks)")

    print("\nTransformer block done! Ready for Chapter 10.")